# Proyecto Final: Simulación de un Sistema M/M/1/K para el Canal de Subida en Redes 5G

## Nombre: Joenny daviel de los santos                                  ID:10143529

## Introducción

Las redes 5G permiten la comunicación de una gran cantidad de dispositivos de manera simultánea, generando un elevado volumen de tráfico de datos. En el canal de subida, los paquetes enviados por los usuarios llegan a la estación base de forma aleatoria y deben ser procesados por un servidor con capacidad limitada. Cuando la demanda supera la capacidad disponible, pueden producirse congestiones, tiempos de espera elevados y pérdida de paquetes.

Para estudiar este comportamiento se utilizan modelos de teoría de colas. En este trabajo se analiza un sistema M/M/1/K, donde las llegadas siguen un proceso de Poisson, los tiempos de servicio tienen distribución exponencial, existe un único servidor y la capacidad de la cola es limitada. Mediante simulación y comparación con resultados analíticos se evalúa el desempeño del sistema bajo diferentes capacidades de servicio y se analiza su impacto sobre la calidad de servicio (QoS) en redes 5G.

## Paso 1: Importación de librerías y configuración de parámetros

En esta primera parte del código se importa la librería NumPy, que se utiliza para realizar cálculos numéricos y generar valores aleatorios. También se definen los parámetros principales de la simulación: la tasa de llegada de paquetes (λ=180 paquetes/s), la capacidad máxima de la cola (K=5) y el número total de llegadas simuladas (N=1,000,000). Además, se establece una semilla aleatoria para garantizar que los resultados puedan reproducirse cada vez que se ejecute el programa.

In [1]:
import numpy as np

# ==========================================
# Parámetros
# ==========================================

LAMBDA = 180
K = 5
N = 1_000_000

np.random.seed(42)

## Paso 2: Simulación del sistema M/M/1/K

La función simular_mm1k() representa el comportamiento del sistema de colas. Primero se generan los tiempos de llegada de los paquetes utilizando una distribución exponencial, característica de un proceso de Poisson. Posteriormente, se simula el funcionamiento del servidor y la cola de espera, verificando en cada llegada si existe espacio disponible para aceptar el paquete.

Si el sistema tiene capacidad disponible, el paquete entra a servicio o espera en la cola hasta ser atendido. En caso de que el sistema se encuentre lleno, el paquete es descartado y se contabiliza como una pérdida. Durante este proceso también se registran los tiempos de espera, el tiempo que el servidor permanece ocupado y la cantidad de paquetes presentes en la cola.

Al finalizar la simulación, se calculan las métricas principales del sistema: probabilidad de bloqueo, utilización del servidor, longitud promedio de la cola y tiempo promedio de espera.

In [2]:
def simular_mm1k(lam, mu, K, N):

    llegadas = np.cumsum(np.random.exponential(1/lam, N))

    cola = []
    bloqueados = 0

    tiempo_ocupado = 0
    area_cola = 0

    tiempo_anterior = 0

    esperas = []

    for llegada in llegadas:

        while cola and cola[0] <= llegada:
            cola.pop(0)

        en_cola = max(0, len(cola) - 1)

        area_cola += en_cola * (llegada - tiempo_anterior)
        tiempo_anterior = llegada

        if len(cola) >= K + 1:
            bloqueados += 1
            continue

        servicio = np.random.exponential(1/mu)

        if len(cola) == 0:
            inicio_servicio = llegada
        else:
            inicio_servicio = cola[-1]

        salida = inicio_servicio + servicio

        cola.append(salida)

        espera = inicio_servicio - llegada
        esperas.append(espera)

        tiempo_ocupado += servicio

    tiempo_total = llegadas[-1]

    p_bloqueo = bloqueados / N

    rho = tiempo_ocupado / tiempo_total

    Lq = area_cola / tiempo_total

    Wq = np.mean(esperas)

    return p_bloqueo, rho, Lq, Wq

## Paso 3: Cálculo analítico del modelo M/M/1/K

La función analitico_mm1k() implementa las ecuaciones teóricas de la teoría de colas para un sistema M/M/1/K. A partir de las tasas de llegada y servicio, se calculan las probabilidades de estado estacionario, la probabilidad de bloqueo, la utilización efectiva del servidor, la longitud promedio de la cola y el tiempo promedio de espera.

Estos resultados sirven como referencia para validar la simulación y comprobar que el comportamiento obtenido experimentalmente coincide con las predicciones matemáticas del modelo.

In [3]:
def analitico_mm1k(lam, mu, K):

    rho = lam / mu

    capacidad_total = K + 1

    P0 = (1-rho)/(1-rho**(capacidad_total+1))

    probs = np.array([
        P0 * rho**n
        for n in range(capacidad_total+1)
    ])

    Pbloqueo = probs[-1]

    L = np.sum(
        np.arange(capacidad_total+1) * probs
    )

    lambda_eff = lam * (1 - Pbloqueo)

    rho_real = lambda_eff / mu

    Lq = L - rho_real

    Wq = Lq / lambda_eff

    return Pbloqueo, rho_real, Lq, Wq

## Paso 4: Evaluación para μ = 200 paquetes/s

En esta etapa se ejecuta la simulación considerando una capacidad de servicio de 200 paquetes por segundo. Posteriormente, se calculan las métricas analíticas correspondientes y se comparan con los resultados obtenidos mediante simulación para verificar la precisión del modelo implementado.

Esta comparación permite evaluar el desempeño del sistema bajo una condición en la que la capacidad de servicio es relativamente cercana a la tasa de llegada de paquetes. En este escenario es más probable que se produzcan congestiones, acumulación de paquetes en la cola y pérdidas debido a la capacidad limitada del búfer. Los resultados obtenidos permiten analizar el comportamiento del sistema cuando opera bajo una carga elevada y determinar si la infraestructura es suficiente para manejar el tráfico generado.

In [4]:
MU = 200

print("\n==============================")
print("M/M/1/K   μ = 200")
print("==============================")

sim200 = simular_mm1k(LAMBDA, MU, K, N)
ana200 = analitico_mm1k(LAMBDA, MU, K)

print("\nSIMULACIÓN")
print(f"Pbloqueo = {sim200[0]:.6f}")
print(f"ρ         = {sim200[1]:.6f}")
print(f"Lq        = {sim200[2]:.6f}")
print(f"Wq        = {sim200[3]:.6f} s")

print("\nANALÍTICO")
print(f"Pbloqueo = {ana200[0]:.6f}")
print(f"ρ         = {ana200[1]:.6f}")
print(f"Lq        = {ana200[2]:.6f}")
print(f"Wq        = {ana200[3]:.6f} s")


M/M/1/K   μ = 200

SIMULACIÓN
Pbloqueo = 0.100913
ρ         = 0.806926
Lq        = 1.227772
Wq        = 0.010910 s

ANALÍTICO
Pbloqueo = 0.101867
ρ         = 0.808320
Lq        = 1.774087
Wq        = 0.010974 s


## Paso 5: Evaluación para μ = 400 paquetes/s

A continuación, se repite el procedimiento utilizando una capacidad de servicio de 400 paquetes por segundo. El objetivo es analizar el impacto de incrementar la velocidad de procesamiento del canal de subida y observar cómo esta mejora afecta el rendimiento general del sistema.

Al disponer de una mayor capacidad de servicio, se espera una reducción significativa en la probabilidad de bloqueo de paquetes, una menor utilización del servidor y tiempos de espera más bajos. Esto permite que los paquetes sean procesados con mayor rapidez, disminuyendo la congestión y mejorando la eficiencia operativa de la red. La comparación entre ambos escenarios facilita la identificación de los beneficios obtenidos al aumentar la capacidad de procesamiento.

In [5]:
MU = 400

print("\n==============================")
print("M/M/1/K   μ = 400")
print("==============================")

sim400 = simular_mm1k(LAMBDA, MU, K, N)
ana400 = analitico_mm1k(LAMBDA, MU, K)

print("\nSIMULACIÓN")
print(f"Pbloqueo = {sim400[0]:.6f}")
print(f"ρ         = {sim400[1]:.6f}")
print(f"Lq        = {sim400[2]:.6f}")
print(f"Wq        = {sim400[3]:.6f} s")

print("\nANALÍTICO")
print(f"Pbloqueo = {ana400[0]:.6f}")
print(f"ρ         = {ana400[1]:.6f}")
print(f"Lq        = {ana400[2]:.6f}")
print(f"Wq        = {ana400[3]:.6f} s")


M/M/1/K   μ = 400

SIMULACIÓN
Pbloqueo = 0.004696
ρ         = 0.448606
Lq        = 0.148401
Wq        = 0.001924 s

ANALÍTICO
Pbloqueo = 0.004584
ρ         = 0.447937
Lq        = 0.343990
Wq        = 0.001920 s


## Paso 6: Análisis de Calidad de Servicio (QoS)

Finalmente, el programa realiza un análisis adicional para determinar la capacidad mínima de servicio necesaria para mantener una probabilidad de bloqueo inferior al 1%, requisito común en sistemas de comunicación que buscan garantizar un servicio confiable. Para ello se prueban distintos valores de μ hasta encontrar el primero que cumple dicha condición.

Este análisis permite identificar la capacidad requerida para garantizar un nivel adecuado de Calidad de Servicio (QoS) en una red 5G bajo la tasa de llegada especificada. Además, proporciona información útil para el dimensionamiento de la infraestructura, ayudando a tomar decisiones sobre la capacidad de procesamiento necesaria para minimizar pérdidas de paquetes, reducir tiempos de espera y ofrecer una mejor experiencia a los usuarios.

In [6]:
print("\n==============================")
print("ANÁLISIS QoS")
print("==============================")

for mu_test in range(200, 501, 10):

    pb, _, _, _ = analitico_mm1k(
        LAMBDA,
        mu_test,
        K
    )

    if pb < 0.01:

        print(
            f"Para K={K}, la capacidad mínima"
        )

        print(
            f"que cumple Pbloqueo < 1% es:"
        )

        print(
            f"μ ≈ {mu_test} paquetes/s"
        )

        print(
            f"Pbloqueo = {pb:.6f}"
        )

        break


ANÁLISIS QoS
Para K=5, la capacidad mínima
que cumple Pbloqueo < 1% es:
μ ≈ 350 paquetes/s
Pbloqueo = 0.009073


## Conclusión

En este trabajo se analizó el comportamiento de un sistema de colas M/M/1/K aplicado al canal de subida de una estación base 5G mediante simulación y comparación con resultados analíticos. Los resultados obtenidos demostraron que la simulación reproduce adecuadamente el comportamiento teórico del sistema, ya que las métricas calculadas presentaron valores muy cercanos a los obtenidos mediante las fórmulas analíticas.

Para una capacidad de servicio de μ = 200 paquetes por segundo, el sistema presentó una probabilidad de bloqueo cercana al 10%, una utilización del servidor superior al 80% y tiempos de espera relativamente mayores, evidenciando un escenario de alta carga y posible congestión. Por otro lado, al incrementar la capacidad de servicio a μ = 400 paquetes por segundo, la probabilidad de bloqueo se redujo a menos del 1%, mientras que la utilización del servidor, la longitud de la cola y los tiempos de espera disminuyeron considerablemente.

Además, el análisis de Calidad de Servicio (QoS) permitió determinar que una capacidad de servicio aproximada de 350 paquetes por segundo es suficiente para mantener una probabilidad de bloqueo inferior al 1% bajo las condiciones planteadas. Esto demuestra que aumentar la capacidad de procesamiento del sistema es una estrategia efectiva para reducir la congestión, minimizar la pérdida de paquetes y mejorar el rendimiento general de la red.

En conclusión, los resultados obtenidos muestran la importancia de dimensionar adecuadamente los recursos de una red 5G para garantizar una transmisión eficiente de datos y ofrecer una mejor experiencia a los usuarios, especialmente en escenarios con altas tasas de tráfico.